# Spleen Segmentation — Train + Evaluate (Colab)

This notebook clones the repo, downloads the MSD Task09_Spleen dataset, trains the 3D U-Net,
evaluates it on a held-out test set, and runs inference on a sample case.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine).

Steps:
1. Setup (clone repo, install deps, mount Drive for persistent checkpoints)
2. Download dataset
3. Data QC
4. Train
5. Evaluate on held-out test set
6. Visualize predictions (best + worst cases)
7. Run inference on a single new volume


In [ ]:
# 1a. Mount Google Drive so checkpoints/reports survive a Colab session reset.
# If you don't want to use Drive, skip this cell and outputs will be lost when the runtime disconnects.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/spleen-segmentation"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f"Persistent storage at: {DRIVE_PROJECT_DIR}")


In [ ]:
# 1b. Clone the repo
!git clone https://github.com/lynda512/Spleen-Segmentation.git /content/Spleen-Segmentation
%cd /content/Spleen-Segmentation


In [ ]:
# 1c. Install dependencies
!pip install -q -r requirements.txt


In [ ]:
# 1d. Point config at Colab/Drive paths via env vars (must be set BEFORE importing src.config)
import os
os.environ["SPLEEN_DATA_DIR"] = "/content/data"
os.environ["SPLEEN_CKPT_DIR"] = f"{DRIVE_PROJECT_DIR}/checkpoints"
os.environ["SPLEEN_REPORTS_DIR"] = f"{DRIVE_PROJECT_DIR}/reports"

import sys
sys.path.insert(0, "/content/Spleen-Segmentation")


In [ ]:
# 1e. Log in to Weights & Biases
# Get your API key from https://wandb.ai/authorize
import wandb
wandb.login()


## 2. Download the dataset

The Medical Segmentation Decathlon Task09_Spleen dataset. The original MSD download links have been
unreliable historically -- if the direct download below fails, the same data is mirrored on Kaggle
(search "Medical Segmentation Decathlon Spleen") or via the MONAI dataset download utility, which
is more robust to link rot. Both options are included below; try Option A first.


In [ ]:
# 2. Option A: download via MONAI's built-in (handles the official MSD Google Drive link + extraction)
from monai.apps import DecathlonDataset

# This downloads + extracts directly into SPLEEN_DATA_DIR/raw/Task09_Spleen, matching what
# src/config.py and src/data/msd_spleen_datamodule.py expect.
import os
from src import config

config.DATA_DIR.mkdir(parents=True, exist_ok=True)
(config.DATA_DIR / "raw").mkdir(parents=True, exist_ok=True)

_ = DecathlonDataset(
    root_dir=str(config.DATA_DIR / "raw"),
    task="Task09_Spleen",
    section="training",
    download=True,
    cache_num=0,  # we only want it on disk here, not cached in memory yet
)
print("Download complete.")
print(list((config.RAW_DIR).iterdir()))


If Option A fails (the official MSD download link has historically been flaky), uncomment and
use Option B below instead -- download the Kaggle mirror and place it at the same path.

In [ ]:
# 2. Option B (fallback): Kaggle mirror
# 1. Upload your kaggle.json API token first (Settings -> API -> Create New Token on kaggle.com)
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d <search-and-fill-in-a-Task09-Spleen-mirror-slug>
# !unzip -q <downloaded-file>.zip -d /content/data/raw/Task09_Spleen


## 3. Data QC\n\nSanity-check the dataset before training: confirms image/label pairing, reports shape/spacing/intensity ranges, and flags whether `INTENSITY_CLIP` in `config.py` is a reasonable window for this data.

In [ ]:
from src.qc.check_dataset import check_dataset
stats = check_dataset()


## 4. Train

Trains the 3D U-Net with Dice+CE loss, sliding-window validation every `VAL_INTERVAL` epochs,
early stopping, and W&B logging. Adjust `--epochs` down if you're short on Colab session time --
even 30-40 epochs should produce a reasonably informative result for a portfolio writeup,
though MONAI's own spleen tutorial trains ~600 epochs for a SOTA number.


In [ ]:
!python -m src.train --epochs 60 --run-name "weekend-v1"


## 5. Evaluate on held-out test set\n\nThis is the honest, unbiased number -- the test set was never used for training or checkpoint selection.

In [ ]:
!python -m src.evaluate --n-worst 3


## 6. Visualize predictions\n\nLook at the best and worst test cases qualitatively -- a low Dice score on a tiny/edge-of-volume spleen often looks different in person than the number suggests.

In [ ]:
import json
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch

from src import config
from src.data.transforms import get_test_transforms
from src.engine import sliding_window_predict
from src.models.unet_3d import build_unet_3d
from src.data.msd_spleen_datamodule import get_datalist_spleen

results = json.loads((config.REPORTS_DIR / "test_results.json").read_text())
print("Mean test Dice:", results["mean_test_dice"])
print("Worst cases:", results["worst_cases"])
print("Best cases:", results["best_cases"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(config.CHECKPOINT_DIR / "best_model.pt", map_location=device)
model = build_unet_3d().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

_, _, test_files = get_datalist_spleen(config.RAW_DIR)
transform = get_test_transforms()

def show_case(case_dict, title):
    sample = transform(case_dict)
    image = sample["image"].unsqueeze(0).to(device)
    label = sample["label"].squeeze(0).numpy()

    with torch.no_grad():
        logits = sliding_window_predict(model, image)
        pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

    mid = image.shape[-1] // 2
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image[0, 0, :, :, mid].cpu(), cmap="gray")
    axes[0].set_title("CT slice")
    axes[1].imshow(label[0, :, :, mid], cmap="gray")
    axes[1].set_title("Ground truth")
    axes[2].imshow(pred[:, :, mid], cmap="gray")
    axes[2].set_title("Prediction")
    fig.suptitle(title)
    plt.show()

# Visualize the single worst case as an example -- loop over results["worst_cases"]
# and match back to test_files by filename if you want all of them.
worst_filename = results["worst_cases"][0][0]
worst_case = next(f for f in test_files if worst_filename in f["image"])
show_case(worst_case, f"Worst case: {worst_filename} (Dice={results['worst_cases'][0][1]:.3f})")


## 7. Run inference on a single new volume\n\nDemonstrates the standalone prediction path (e.g. for a volume with no ground truth).

In [ ]:
# Using one of the test volumes as a stand-in for a "new" unlabeled scan
sample_input = test_files[0]["image"]
!python -m src.predict --input "{sample_input}" --output /content/sample_prediction.nii.gz


## Next steps

- Bump epochs and re-run if Dice is still climbing at the end of training (check the W&B loss/Dice curves)
- Try Hausdorff distance / surface distance as a secondary metric alongside Dice
- Resample predictions back to native image space in `predict.py` (currently outputs in resampled space -- see the note printed by the script)
- See main README.md "What's next" section for the longer roadmap
